<!-- paired-role-banner -->
> 🟧 **실습 코드 · 왼쪽 창** — `TODO`/빈 구현을 직접 완성하세요. 정답을 보기 전에 입력·출력 shape를 먼저 예측하세요.

# 14. RNN/GRU/LSTM 시퀀스 모델링

센서 시간 창을 이용해 **이상 징후 분류**와 **다음 온도 예측**을 수행합니다.
vanilla RNN, GRU, LSTM의 인터페이스를 같은 클래스로 비교하고, 가변 길이 padding,
`pack_padded_sequence`, hidden state shape, gradient clipping, checkpoint까지 연습합니다.

In [ ]:
# 이 컴퓨터를 감지해 안전한 시작값을 확인합니다. 각 실습은 더 작은 값으로도 동작합니다.
from llm_engineering_lab.hardware import HardwareProfile, recommend_training_profile

LAB_HARDWARE = HardwareProfile.detect()
LAB_PROFILE = recommend_training_profile(LAB_HARDWARE)
print("hardware:", LAB_HARDWARE)
print("recommended profile:", LAB_PROFILE)

## 실행 규모

기본 설정은 CPU에서도 수 분 이내, RTX 3080에서는 더 빠르게 끝나도록 작은 모델과
적은 epoch를 사용합니다. GPU 메모리보다 데이터 전송 시간이 더 큰 규모이므로
`device` 코드는 연습하되, CPU가 반드시 열등한 선택은 아닙니다.

In [ ]:
import copy
import random
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score, mean_absolute_error
from torch import nn
from torch.nn.utils import clip_grad_norm_
from torch.nn.utils.rnn import pack_padded_sequence, pad_sequence
from torch.utils.data import DataLoader, Dataset
from llm_engineering_lab.acceleration import get_accelerator

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
ACCELERATOR = get_accelerator()
device = ACCELERATOR.device
print(ACCELERATOR.summary(), "torch:", torch.__version__)

In [ ]:
from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "data" / "practice").exists():
            return candidate
    raise FileNotFoundError(
        "프로젝트 루트를 찾지 못했습니다. Start_JupyterLab.cmd로 실행해 주세요."
    )


ROOT = find_project_root()
DATA = ROOT / "data" / "practice"
ARTIFACTS = ROOT / "artifacts" / "practice"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
print("project root:", ROOT)

## 1) 시간 순서 보존과 train 통계

장비별 앞 75%를 train, 뒤 25%를 test로 두고, 표준화 통계는 train 구간만으로
계산합니다. target window 안에 anomaly가 하나라도 있으면 양성으로 정의합니다.

In [ ]:
frame = pd.read_csv(DATA / "sensor_timeseries.csv", parse_dates=["timestamp"])
FEATURES = ["load", "temperature", "vibration", "pressure"]

train_parts, test_parts = [], []
for _, group in frame.sort_values("timestamp").groupby("device_id", sort=False):
    cut = int(len(group) * 0.75)
    train_parts.append(group.iloc[:cut])
    test_parts.append(group.iloc[cut:])
train_frame = pd.concat(train_parts, ignore_index=True)
test_frame = pd.concat(test_parts, ignore_index=True)
means = train_frame[FEATURES].mean()
stds = train_frame[FEATURES].std().replace(0, 1.0)
print(train_frame.shape, test_frame.shape, "raw anomaly rate=", frame.anomaly.mean())

In [ ]:
def build_variable_sequences(frame, means, stds, max_history=12, stride=3):
    # TODO 14-1: train 통계로 표준화하고, 장비 경계를 넘지 않는
    # 길이 6~12의 Tensor 목록과 window anomaly max label을 만드세요.
    raise NotImplementedError


train_sequences, train_labels = build_variable_sequences(train_frame, means, stds)
test_sequences, test_labels = build_variable_sequences(test_frame, means, stds)

## 2) padding과 원래 길이

padding은 배치 텐서를 직사각형으로 만들 뿐 실제 관측값이 아닙니다. 원래 길이를
보존한 뒤 packing하거나 mask해야 모델이 padding을 신호로 학습하지 않습니다.

In [ ]:
class SequenceDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences, self.labels = sequences, labels

    def __len__(self):
        # TODO 14-2a
        raise NotImplementedError

    def __getitem__(self, index):
        # TODO 14-2b
        raise NotImplementedError


def sequence_collate(batch):
    # TODO 14-2c: batch_first padding, lengths, stacked labels를 반환하세요.
    raise NotImplementedError


train_loader = DataLoader(
    SequenceDataset(train_sequences, train_labels),
    batch_size=64,
    shuffle=True,
    collate_fn=sequence_collate,
)
test_loader = DataLoader(
    SequenceDataset(test_sequences, test_labels),
    batch_size=128,
    shuffle=False,
    collate_fn=sequence_collate,
)
batch_x, batch_lengths, batch_y = next(iter(train_loader))

In [ ]:
assert batch_x.ndim == 3 and batch_x.shape[2] == len(FEATURES)
assert int(batch_lengths.max()) == batch_x.shape[1]
assert torch.all((batch_x.abs().sum(dim=2) != 0).sum(dim=1) <= batch_lengths)
print("padded batch contract: OK")

## 3) 하나의 wrapper로 RNN/GRU/LSTM 비교

세 모듈은 공통적으로 `(output, hidden)`을 반환합니다. LSTM만 hidden이 `(h, c)`
튜플이며, 다층/양방향 모델의 `h` shape는 `(layers × directions, batch, hidden)`입니다.

In [ ]:
class SequenceClassifier(nn.Module):
    def __init__(
        self, cell: str, input_dim: int, hidden_dim: int = 24, num_layers: int = 1
    ):
        super().__init__()
        # TODO 14-3a: cell 문자열로 nn.RNN/GRU/LSTM을 선택하고 head를 만드세요.
        raise NotImplementedError

    def forward(self, x, lengths, return_hidden=False):
        # TODO 14-3b: pack → recurrent → 마지막 layer h → linear 순서로 구현하세요.
        # LSTM의 hidden tuple도 처리해야 합니다.
        raise NotImplementedError

In [ ]:
for cell in ("RNN", "GRU", "LSTM"):
    probe = SequenceClassifier(cell, len(FEATURES), hidden_dim=16)
    logits, hidden = probe(batch_x[:5], batch_lengths[:5], return_hidden=True)
    if isinstance(hidden, tuple):
        hidden_shapes = [tuple(item.shape) for item in hidden]
    else:
        hidden_shapes = tuple(hidden.shape)
    print(cell, "logits", tuple(logits.shape), "hidden", hidden_shapes)
    assert logits.shape == (5,)

## 4) gradient clipping을 포함한 학습 루프

긴 시퀀스의 반복 곱은 gradient exploding을 일으킬 수 있습니다. 역전파 후
optimizer step 전에 `clip_grad_norm_`을 호출하고, 반환되는 clipping 전 norm을 기록합니다.

In [ ]:
@dataclass
class EpochStats:
    loss: float
    grad_norm: float


def train_epoch(model, loader, optimizer, loss_fn, max_norm=1.0):
    # TODO 14-4a: forward/backward → clip_grad_norm_ → optimizer.step을 구현하세요.
    raise NotImplementedError


@torch.inference_mode()
def evaluate_classifier(model, loader):
    # TODO 14-4b: sigmoid 확률, F1, labels를 반환하세요.
    raise NotImplementedError

In [ ]:
positive = float(train_labels.sum())
negative = float(len(train_labels) - positive)
pos_weight = torch.tensor([negative / max(positive, 1.0)], device=device)
classification_loss = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
print("pos_weight:", round(pos_weight.item(), 2))

In [ ]:
comparison = {}
trained_models = {}
# TODO 14-5: RNN/GRU/LSTM 각각 3 epoch 학습하고 loss와 test F1을 저장하세요.
# 힌트: model → Adam → train_epoch → evaluate_classifier 순서입니다.
raise NotImplementedError

In [ ]:
result_table = pd.DataFrame(comparison).T.sort_values("test_f1", ascending=False)
print(result_table)
assert set(result_table.index) == {"RNN", "GRU", "LSTM"}
assert np.isfinite(result_table.to_numpy()).all()

### 비교를 해석할 때 주의할 점

작은 데이터에서 한 번의 seed로 얻은 순위는 모델의 보편적 우열이 아닙니다.
RNN은 단순하지만 장기 의존성에 취약하고, GRU는 gate가 적어 가벼우며, LSTM은
cell state를 별도로 유지합니다. 반복 실험의 평균·편차와 latency를 함께 비교하세요.

## 5) padding 값 불변성 확인

packing이 올바르면 실제 길이 뒤의 padding 값을 크게 바꿔도 logits는 변하지 않습니다.
이 테스트는 mask/length 누락 버그를 매우 잘 찾아냅니다.

In [ ]:
best_name = result_table.index[0]
best_model = trained_models[best_name].eval()
x = batch_x[:8].to(device)
lengths = batch_lengths[:8]
changed_padding = x.clone()
for row, length in enumerate(lengths.tolist()):
    changed_padding[row, length:] = 999.0
with torch.inference_mode():
    original_logits = best_model(x, lengths)
    changed_logits = best_model(changed_padding, lengths)
max_difference = (original_logits - changed_logits).abs().max().item()
print("max padding-induced difference:", max_difference)
assert max_difference < 1e-5

## 6) checkpoint 저장과 복원

최소 checkpoint에는 모델 종류/차원, `state_dict`, 전처리 통계, feature 순서가 있어야
합니다. optimizer 상태와 epoch까지 저장하면 중단된 학습도 이어갈 수 있습니다.

In [ ]:
checkpoint_path = ARTIFACTS / "rnn_sequence_classifier_exercise.pt"
# TODO 14-6: model 설정, state_dict, features, train 통계를 저장하고
# 새 SequenceClassifier에 복원한 뒤 logits가 같은지 확인하세요.
raise NotImplementedError

## 7) 다음 온도 forecasting

분류와 달리 연속값을 예측하므로 MSE로 학습하고 MAE로 평가합니다. 각 window 직후의
`next_temperature`를 target으로 하며, target 역시 train 평균/표준편차로 정규화합니다.

In [ ]:
def make_forecast_samples(frame, feature_means, feature_stds, history=12, stride=3):
    # TODO 14-7: 고정 길이 입력 window와 정규화한 next_temperature target을 만드세요.
    # target 통계도 train_frame에서만 계산해야 합니다.
    raise NotImplementedError


forecast_train_x, forecast_train_y, target_mean, target_std = make_forecast_samples(
    train_frame, means, stds
)
forecast_test_x, forecast_test_y, _, _ = make_forecast_samples(test_frame, means, stds)

In [ ]:
class GRUForecaster(nn.Module):
    def __init__(self, input_dim, hidden_dim=24):
        super().__init__()
        # TODO 14-8a: GRU와 회귀 head를 만드세요.
        raise NotImplementedError

    def forward(self, x):
        # TODO 14-8b: 마지막 hidden으로 연속값 하나를 예측하세요.
        raise NotImplementedError


# TODO 14-8c: MSE, Adam, gradient clipping으로 4 epoch 학습하세요.
raise NotImplementedError

In [ ]:
forecast_model.eval()
with torch.inference_mode():
    normalized_prediction = forecast_model(forecast_test_x.to(device)).cpu().numpy()
predicted_temperature = normalized_prediction * target_std + target_mean
actual_temperature = forecast_test_y.numpy() * target_std + target_mean
forecast_mae = mean_absolute_error(actual_temperature, predicted_temperature)
print("test MAE (°C):", round(forecast_mae, 3))
for actual, predicted in list(zip(actual_temperature, predicted_temperature))[:5]:
    print(f"actual={actual:6.2f}  predicted={predicted:6.2f}")
assert np.isfinite(forecast_mae)

## hidden state shape 기억법

- 입력(`batch_first=True`): `(batch, time, input_dim)`
- output: `(batch, time, directions × hidden_dim)`
- h: `(layers × directions, batch, hidden_dim)`
- LSTM: `(h, c)` 두 텐서를 반환
- packed 입력에서도 hidden은 각 샘플의 **실제 마지막 timestep**을 반영

In [ ]:
summary = {
    "classification_models": list(comparison),
    "best_cell": best_name,
    "padding_invariance": max_difference,
    "forecast_mae": float(forecast_mae),
    "checkpoint": str(checkpoint_path),
}
print(summary)
assert checkpoint_path.exists()
assert summary["padding_invariance"] < 1e-5

## 확장 과제

1. `num_layers=2`, dropout, bidirectional을 추가하고 hidden shape를 기록하세요.
2. 분류 threshold를 validation F1로 선택하고 test에는 한 번만 적용하세요.
3. forecasting을 한 시점이 아니라 다음 6개 시점 출력으로 바꾸세요.
4. gradient norm을 epoch별로 기록하여 clipping 전후 안정성을 비교하세요.

## 마무리

시퀀스 모델의 핵심은 셀 이름보다 **시간 분할, 길이 전달, hidden 선택, gradient 안정성,
재현 가능한 checkpoint**입니다. Transformer에서도 padding mask와 시간 누수 원칙은 그대로 이어집니다.